In [ ]:
# #1: Cai dat
# !pip install transformers evaluate jiwer wandb -q

In [ ]:
#2: Import
import os, gc, json, time, glob
import numpy as np
import torch
import wandb
import psutil
from dataclasses import dataclass
from typing import List, Dict, Union
from torch.utils.data import Dataset
from transformers import (
    Wav2Vec2ForCTC,
    Wav2Vec2Processor,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    EarlyStoppingCallback,
)
from evaluate import load as load_metric

In [ ]:
#3: Environment
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

def print_memory(label=''):
    ram = psutil.virtual_memory()
    if torch.cuda.is_available():
        vram = torch.cuda.memory_allocated() / 1e9
        vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'[{label}] RAM: {ram.used/1e9:.1f}/{ram.total/1e9:.1f}GB | VRAM: {vram:.1f}/{vram_total:.1f}GB')
    else:
        print(f'[{label}] RAM: {ram.used/1e9:.1f}/{ram.total/1e9:.1f}GB')

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')
else:
    print('Không có GPU!')
print_memory('start')

In [ ]:
#4: Config
PROCESSED_DIR = '/kaggle/input/datasets/thientmai220205/vimd-dataset-wav2vec'
OUTPUT_DIR    = '/kaggle/working/wav2vec2-vimd-checkpoint'
BEST_DIR      = '/kaggle/working/wav2vec2-vimd-bestmodel'

MODEL_NAME    = 'nguyenvulebinh/wav2vec2-base-vietnamese-250h'

TRAIN_BATCH   = 4
EVAL_BATCH    = 8
GRAD_ACCUM    = 4
NUM_EPOCHS    = 25
LEARNING_RATE = 1e-4

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(BEST_DIR,   exist_ok=True)

print(f'Config ready!')
print(f'Train batch: {TRAIN_BATCH} x {GRAD_ACCUM} = {TRAIN_BATCH*GRAD_ACCUM} effective')
print(f'Epochs: {NUM_EPOCHS}')
print(f'PROCESSED_DIR: {PROCESSED_DIR}')

In [ ]:
#5: WandB
wandb.login(key='')
run = wandb.init(
    project='',
    entity='',
    name='',
    resume='allow',
)

In [ ]:
#6: Load Processor
from huggingface_hub import hf_hub_download
from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor

vocab_file = hf_hub_download(repo_id=MODEL_NAME, filename='vocab.json')
with open(vocab_file, 'r', encoding='utf-8') as f:
    vocab_dict = json.load(f)
print(f'Vocab goc: {len(vocab_dict)} tokens')

VOCAB_PATH = '/kaggle/working/vocab.json'
with open(VOCAB_PATH, 'w', encoding='utf-8') as f:
    json.dump(vocab_dict, f, ensure_ascii=False, indent=2)

tokenizer = Wav2Vec2CTCTokenizer(
    VOCAB_PATH,
    unk_token='<unk>',
    pad_token='<pad>',
    word_delimiter_token='|',
    bos_token=None,
    eos_token=None,
)
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1, sampling_rate=16000,
    padding_value=0.0, do_normalize=True, return_attention_mask=False,
)
processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)
print(f'Vocab size: {len(processor.tokenizer)}')
print(f'PAD id: {processor.tokenizer.pad_token_id}')
print_memory('after processor')

In [ ]:
#7: Load VIMD .npy
def load_npy_split(processed_dir, split='train'):
    iv_files = sorted(glob.glob(f'{processed_dir}/{split}_iv_*.npy'))
    lb_files = sorted(glob.glob(f'{processed_dir}/{split}_lb_*.npy'))
    if not iv_files:
        raise FileNotFoundError(f'Không tìm thấy {split}_iv_*.npy tai {processed_dir}')
    print(f'  Loading {len(iv_files)} batch files cho {split}...')
    all_iv = np.concatenate([np.load(f, allow_pickle=True) for f in iv_files])
    all_lb = np.concatenate([np.load(f, allow_pickle=True) for f in lb_files])
    print(f'VIMD {split}: {len(all_iv):,} samples')
    return all_iv, all_lb

print('Loading VIMD train...')
train_iv, train_lb = load_npy_split(PROCESSED_DIR, 'train')
print_memory('after train')

print('Loading VIMD val...')
val_iv, val_lb = load_npy_split(PROCESSED_DIR, 'val')
print_memory('after val')

# test: dùng để chọn best checkpoint
print('Loading VIMD test...')
test_iv, test_lb = load_npy_split(PROCESSED_DIR, 'test')
print_memory('after test')

print(f'\nTrain: {len(train_iv):,} | Val: {len(val_iv):,} | Test: {len(test_iv):,}')

In [ ]:
#8: PyTorch Dataset
class VIMDNpyDataset(Dataset):
    def __init__(self, input_values, labels):
        self.input_values = input_values
        self.labels       = labels

    def __len__(self):
        return len(self.input_values)

    def __getitem__(self, idx):
        return {
            'input_values': np.array(self.input_values[idx], dtype=np.float32),
            'labels':       [int(x) for x in self.labels[idx]],
        }

train_dataset = VIMDNpyDataset(train_iv, train_lb)
val_dataset   = VIMDNpyDataset(val_iv,   val_lb)
test_dataset  = VIMDNpyDataset(test_iv,  test_lb)

print(f'Train: {len(train_dataset):,} | Val: {len(val_dataset):,} | Test: {len(test_dataset):,}')
print_memory('after datasets')

In [ ]:
#9: Load Model
print_memory('before model')
model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    attention_dropout=0.1,
    hidden_dropout=0.1,
    feat_proj_dropout=0.0,
    mask_time_prob=0.05,
    layerdrop=0.1,
    ctc_loss_reduction='mean',
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
    ctc_zero_infinity=True,
)
model.freeze_feature_encoder()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')
print_memory('after model')

In [ ]:
#10: Data Collator
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        input_features = [{'input_values': f['input_values']} for f in features]
        label_features = [{'input_ids': f['labels']}          for f in features]
        batch = self.processor.pad(input_features, padding=self.padding, return_tensors='pt')
        labels_batch = self.processor.tokenizer.pad(label_features, padding=self.padding, return_tensors='pt')
        labels = labels_batch['input_ids'].masked_fill(labels_batch.attention_mask.ne(1), -100)
        labels = labels.clone()
        labels[labels >= len(processor.tokenizer)] = -100
        batch['labels'] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

In [ ]:
#11: Metrics
wer_metric = load_metric('wer')
cer_metric = load_metric('cer')

def compute_metrics(pred):
    pred_ids = np.argmax(pred.predictions, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.batch_decode(pred_ids)
    label_str = processor.tokenizer.batch_decode(pred.label_ids, skip_special_tokens=True)
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    return {'wer': wer, 'cer': cer}

In [ ]:
#12: Callbacks
finetune_start = time.time()
best_test_wer  = float('inf')
epoch_history  = []

def eval_full(dataset, name, device):
    preds, refs, times, audio_lens = [], [], [], []
    total_loss = 0.0
    for i in range(len(dataset)):
        sample       = dataset[i]
        input_values = torch.tensor(sample['input_values'], dtype=torch.float32).unsqueeze(0).to(device)
        labels       = torch.tensor(sample['labels'], dtype=torch.long).unsqueeze(0).to(device)
        labels[labels >= len(processor.tokenizer)] = -100
        audio_lens.append(len(sample['input_values']) / 16000)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            output = model(input_values=input_values, labels=labels)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
        if output.loss is not None:
            total_loss += output.loss.item()
        pred_ids  = torch.argmax(output.logits, dim=-1)
        pred_str  = processor.batch_decode(pred_ids)[0]
        label_ids = [int(x) for x in sample['labels'] if int(x) != -100]
        label_str = processor.tokenizer.decode(label_ids)
        preds.append(pred_str)
        refs.append(label_str)
        if (i + 1) % 500 == 0:
            print(f'    [{i+1}/{len(dataset)}]')
    wer    = wer_metric.compute(predictions=preds, references=refs)
    cer    = cer_metric.compute(predictions=preds, references=refs)
    avg_ms = np.mean(times) * 1000
    loss   = total_loss / len(dataset)
    return wer, cer, avg_ms, loss, preds, refs


class FullEvalCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        global best_test_wer
        epoch  = int(state.epoch)
        device = next(model.parameters()).device
        model.eval()

        train_loss = next(
            (log['loss'] for log in reversed(state.log_history) if 'loss' in log), None
        )

        print(f'\nEpoch {epoch} - Evaluating val ({len(val_dataset):,})...')
        v_wer, v_cer, v_ms, v_loss, v_preds, v_refs = eval_full(val_dataset,  'val',  device)

        print(f'Epoch {epoch} - Evaluating test ({len(test_dataset):,})...')
        t_wer, t_cer, t_ms, t_loss, t_preds, t_refs = eval_full(test_dataset, 'test', device)

        elapsed = (time.time() - finetune_start) / 3600

        #Best model theo TEST WER
        is_best = t_wer < best_test_wer
        if is_best:
            best_test_wer = t_wer
            model.save_pretrained(BEST_DIR)
            processor.save_pretrained(BEST_DIR)

        epoch_history.append({
            'epoch': epoch, 'train_loss': train_loss,
            'val_loss':  v_loss,  'val_wer':  v_wer,  'val_cer':  v_cer,  'val_ms':  v_ms,
            'test_loss': t_loss,  'test_wer': t_wer,  'test_cer': t_cer,  'test_ms': t_ms,
        })

        print(f"\n{'='*70}")
        print(f'Epoch {epoch:2d} | Elapsed: {elapsed:.2f}h {"BEST TEST WER" if is_best else ""}')
        print(f"{'─'*70}")
        print(f"  {'Metric':<22} {'Val':>12} {'Test':>12}")
        print(f"  {'─'*48}")
        if train_loss:
            print(f"  {'Train Loss':<22} {train_loss:>12.4f}")
        print(f"  {'Val Loss':<22} {v_loss:>12.4f}")
        print(f"  {'Test Loss':<22} {t_loss:>12.4f}")
        print(f"  {'─'*48}")
        print(f"  {'WER':<22} {v_wer*100:>11.2f}% {t_wer*100:>11.2f}% {'<- best' if is_best else ''}")
        print(f"  {'CER':<22} {v_cer*100:>11.2f}% {t_cer*100:>11.2f}%")
        print(f"  {'Avg Infer (ms)':<22} {v_ms:>12.1f} {t_ms:>12.1f}")
        print(f"  {'─'*48}")
        print(f"  Best test WER so far: {best_test_wer*100:.2f}%")
        print(f"\n  Sample predictions:")
        print(f"  [Val]   True: '{v_refs[0]}'")
        print(f"          Pred: '{v_preds[0]}'")
        print(f"  [Test] True: '{t_refs[0]}'")
        print(f"          Pred: '{t_preds[0]}'")
        print(f"{'='*70}\n")

        log_dict = {
            'epoch':           epoch,
            'val/loss':        v_loss,  'val/wer':   v_wer,  'val/cer':   v_cer,  'val/infer_ms':   v_ms,
            'test/loss':      t_loss,  'test/wer': t_wer,  'test/cer': t_cer,  'test/infer_ms': t_ms,
            'best/test_wer':  best_test_wer,
            'elapsed_hours':   elapsed,
        }
        if train_loss:
            log_dict['train/loss'] = train_loss
        wandb.log(log_dict)
        model.train()


class CheckpointCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        try:
            ckpt_dirs = sorted([d for d in os.listdir(args.output_dir) if d.startswith('checkpoint')])
            if ckpt_dirs:
                print(f'Checkpoint saved: {ckpt_dirs[-1]}')
        except Exception as e:
            print(f'Warning: {e}')

print('Callbacks ready!')

In [ ]:
#13: Training Arguments
USE_FP16 = torch.cuda.is_available()

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH,
    per_device_eval_batch_size=EVAL_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='wer',
    greater_is_better=False,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='cosine',
    weight_decay=0.01,
    adam_epsilon=1e-8,
    max_grad_norm=1.0,
    fp16=USE_FP16,
    bf16=False,
    remove_unused_columns=False,
    ddp_find_unused_parameters=False,
    logging_steps=50,
    report_to='wandb',
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    run_name='wav2vec2-vimd-kaggle-t4',
)

In [ ]:
# Cell 14: Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        FullEvalCallback(),
        CheckpointCallback(),
        EarlyStoppingCallback(
            early_stopping_patience=5,
            early_stopping_threshold=0.001,
        ),
    ],
)

In [ ]:
# 15: Train
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print_memory('before training')

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f'Training tren {gpu_name}...')
print(f'Train: {len(train_dataset):,} | Val: {len(val_dataset):,} | Test: {len(test_dataset):,}')
print(f'Epochs: {NUM_EPOCHS} | Effective batch: {TRAIN_BATCH*GRAD_ACCUM}')
print(f'Best model: TEST LOSS thap nhat')

resume = True if os.path.exists(OUTPUT_DIR) and any(
    d.startswith('checkpoint') for d in os.listdir(OUTPUT_DIR)
) else None
print('Resume từ checkpoint...' if resume else 'Start từ đầu...')

train_start = time.time()
trainer.train(resume_from_checkpoint=resume)
train_end   = time.time()

TOTAL_TRAINING_TIME = train_end - train_start
print(f'Finetune time: {TOTAL_TRAINING_TIME/3600:.2f} hours')

In [ ]:
#16: Save model
processor.save_pretrained(BEST_DIR)
print(f'Best model: {BEST_DIR}')
print(f'Best test WER: {best_test_wer*100:.2f}%')

In [ ]:
#17: Epoch History Summary
print(f"{'='*80}")
print('EPOCH HISTORY SUMMARY')
print(f"{'─'*80}")
print(f"  {'Epoch':>5} {'Train Loss':>12} {'Val Loss':>10} {'Val WER':>9} {'Test Loss':>12} {'Test WER':>10} {'Infer(ms)':>10}")
print(f"  {'─'*75}")

best_epoch = min(epoch_history, key=lambda x: x['test_wer'])

for h in epoch_history:
    marker     = ' <- BEST' if h['test_wer'] == best_epoch['test_wer'] else ''
    train_loss = f"{h['train_loss']:.4f}" if h['train_loss'] else 'N/A'
    print(
        f"  {h['epoch']:>5} {train_loss:>12} {h['val_loss']:>10.4f} "
        f"{h['val_wer']*100:>8.2f}% {h['test_loss']:>12.4f} "
        f"{h['test_wer']*100:>9.2f}% {h['test_ms']:>9.1f}{marker}"
    )

print(f"{'─'*80}")
print(f"Best epoch (test WER): {best_epoch['epoch']}")
print(f"  Test WER:   {best_epoch['test_wer']*100:.2f}%")
print(f"  Test CER:   {best_epoch['test_cer']*100:.2f}%")
print(f"  Test Loss:  {best_epoch['test_loss']:.4f}")
print(f"  Val WER:     {best_epoch['val_wer']*100:.2f}%")
print(f"  Val Loss:    {best_epoch['val_loss']:.4f}")
print(f"  Infer (ms):  {best_epoch['test_ms']:.1f} ms/sample")
print(f"  Finetune:    {TOTAL_TRAINING_TIME/3600:.2f} hours")
print(f"  GPU:         {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"{'='*80}")
print(f"\nBest model da luu tai: {BEST_DIR}")
print(f"-> Su dung best model nay de danh gia voi TEST2 doc lap!")

wandb.log({
    'final/best_epoch':      best_epoch['epoch'],
    'final/best_test_wer':  best_epoch['test_wer'],
    'final/best_test_cer':  best_epoch['test_cer'],
    'final/best_test_loss': best_epoch['test_loss'],
    'final/best_val_wer':    best_epoch['val_wer'],
    'final/best_val_cer':    best_epoch['val_cer'],
    'final/infer_ms':        best_epoch['test_ms'],
    'final/finetune_hours':  TOTAL_TRAINING_TIME / 3600,
})
wandb.finish()
print('Hoàn thành')